In [ ]:
import sys
import os
from pyspark import SparkContext, SparkConf
from pyspark.streaming import StreamingContext

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("TrafficStreamingAnalysis").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

ssc = StreamingContext(sc, 5)

input_directory = "file:///home/reza/HW4/stream_input"
dstream = ssc.textFileStream(input_directory)

def parse_traffic_line(line):
    try:
        parts = line.split(',')
        device_code = parts[0].strip()
        pass_time = parts[4].strip()
        plate_index = parts[6].strip()
        return (device_code, pass_time, plate_index)
    except:
        return None

parsed_dstream = dstream.map(parse_traffic_line).filter(lambda x: x is not None)

parsed_dstream.pprint()

ssc.start()

In [ ]:
import sys
import os
from pyspark import SparkContext, SparkConf
from pyspark.streaming import StreamingContext

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("TrafficWindowAnalysis").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

ssc = StreamingContext(sc, 5)

input_directory = "file:///home/reza/HW4/stream_input"
dstream = ssc.textFileStream(input_directory)

def parse_traffic_line(line):
    try:
        parts = line.split(',')
        device_code = parts[0].strip()
        pass_time = parts[4].strip()
        plate_index = parts[6].strip()
        return (device_code, pass_time, plate_index)
    except:
        return None

parsed_dstream = dstream.map(parse_traffic_line).filter(lambda x: x is not None)

WINDOW_DURATION = 30
SLIDE_DURATION = 10

windowed_dstream = parsed_dstream.window(WINDOW_DURATION, SLIDE_DURATION)

def print_total_count(rdd):
    count = rdd.count()
    if count > 0:
        print(f"--- Total Vehicles in last {WINDOW_DURATION} seconds: {count} ---")

windowed_dstream.foreachRDD(print_total_count)

per_camera_counts = windowed_dstream.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)

sorted_camera_counts = per_camera_counts.transform(lambda rdd: rdd.sortBy(lambda x: x[1], ascending=False))

sorted_camera_counts.pprint(10)

ssc.start()

In [ ]:
import sys
import os
from pyspark import SparkContext, SparkConf
from pyspark.streaming import StreamingContext

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("PeakTrafficAnalysis").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

ssc = StreamingContext(sc, 5)

input_directory = "file:///home/reza/HW4/stream_input"
dstream = ssc.textFileStream(input_directory)

def parse_traffic_line(line):
    try:
        parts = line.split(',')
        device_code = parts[0].strip()
        pass_time = parts[4].strip()
        plate_index = parts[6].strip()
        return (device_code, pass_time, plate_index)
    except:
        return None

parsed_dstream = dstream.map(parse_traffic_line).filter(lambda x: x is not None)

WINDOW_DURATION = 30
SLIDE_DURATION = 10

windowed_dstream = parsed_dstream.window(WINDOW_DURATION, SLIDE_DURATION)

def process_rdd(time, rdd):
    try:
        if not rdd.isEmpty():
            top_3 = rdd.take(3)
            print(f"========= Time: {time} =========")
            print(f"Top 3 Busiest Cameras in last {WINDOW_DURATION} seconds:")
            for rank, (camera, count) in enumerate(top_3, 1):
                print(f"{rank}. Camera {camera}: {count} vehicles")
    except:
        pass

per_camera_counts = windowed_dstream.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)

sorted_camera_counts = per_camera_counts.transform(lambda rdd: rdd.sortBy(lambda x: x[1], ascending=False))

sorted_camera_counts.foreachRDD(process_rdd)

ssc.start()

In [ ]:
import sys
import os
from pyspark import SparkContext, SparkConf
from pyspark.streaming import StreamingContext

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("FrequentCarAnalysis").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

ssc = StreamingContext(sc, 5)

input_directory = "file:///home/reza/HW4/stream_input"
dstream = ssc.textFileStream(input_directory)

def parse_plate(line):
    try:
        parts = line.split(',')
        plate_index = parts[6].strip()
        return plate_index
    except:
        return None

plate_dstream = dstream.map(parse_plate).filter(lambda x: x is not None)

WINDOW_DURATION = 60
SLIDE_DURATION = 10
THRESHOLD_N = 3

windowed_plates = plate_dstream.window(WINDOW_DURATION, SLIDE_DURATION)

def process_frequent_cars(time, rdd):
    try:
        if not rdd.isEmpty():
            frequent_cars = rdd.map(lambda x: (x, 1)) \
                               .reduceByKey(lambda a, b: a + b) \
                               .filter(lambda x: x[1] > THRESHOLD_N) \
                               .sortBy(lambda x: x[1], ascending=False) \
                               .take(20)
            
            print(f"========= Time: {time} =========")
            print(f"Cars seen more than {THRESHOLD_N} times in last {WINDOW_DURATION} seconds:")
            for rank, (plate, count) in enumerate(frequent_cars, 1):
                print(f"{rank}. Plate {plate}: {count} sightings")
    except:
        pass

windowed_plates.foreachRDD(process_frequent_cars)

ssc.start()

In [ ]:
import sys
import os
from pyspark import SparkContext, SparkConf
from pyspark.streaming import StreamingContext

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("DiversityAnalysis").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

ssc = StreamingContext(sc, 5)

input_directory = "file:///home/reza/HW4/stream_input"
dstream = ssc.textFileStream(input_directory)

def parse_device_plate(line):
    try:
        parts = line.split(',')
        device_code = parts[0].strip()
        plate_index = parts[6].strip()
        return (device_code, plate_index)
    except:
        return None


device_plate_dstream = dstream.map(parse_device_plate).filter(lambda x: x is not None)

WINDOW_DURATION = 60
SLIDE_DURATION = 20
windowed_dstream = device_plate_dstream.window(WINDOW_DURATION, SLIDE_DURATION)

def process_diversity(time, rdd):
    try:
        if not rdd.isEmpty():
            total_counts = rdd.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)
            unique_counts = rdd.distinct().map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)
            stats = total_counts.join(unique_counts)
            results = stats.mapValues(lambda x: (x[0], x[1], x[1]/x[0])) \
                           .map(lambda x: (x[0], x[1][0], x[1][1], x[1][2])) \
                           .sortBy(lambda x: x[1], ascending=False) \
                           .take(15)
            
            print(f"========= Time: {time} =========")
            print(f"{'Camera':<15} | {'Total Passes':<15} | {'Unique Cars':<15} | {'Diversity Ratio':<15}")
            print("-" * 70)
            for cam, total, unique, ratio in results:
                print(f"{cam:<15} | {total:<15} | {unique:<15} | {ratio:.4f}")
    except:
        pass

windowed_dstream.foreachRDD(process_diversity)

ssc.start()

In [ ]:
import sys
import os
from pyspark import SparkContext, SparkConf
from pyspark.streaming import StreamingContext

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("CameraBehaviorComparison").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

ssc = StreamingContext(sc, 5)

input_directory = "file:///home/reza/HW4/stream_input"
dstream = ssc.textFileStream(input_directory)

def parse_device_plate(line):
    try:
        parts = line.split(',')
        device_code = parts[0].strip()
        plate_index = parts[6].strip()
        return (device_code, plate_index)
    except:
        return None

device_plate_dstream = dstream.map(parse_device_plate).filter(lambda x: x is not None)


WINDOW_DURATION = 60
SLIDE_DURATION = 20
windowed_dstream = device_plate_dstream.window(WINDOW_DURATION, SLIDE_DURATION)

MIN_TRAFFIC_COUNT = 100   
MAX_DIVERSITY_RATIO = 0.60 

def analyze_camera_behavior(time, rdd):
    try:
        if not rdd.isEmpty():
          
            total_counts = rdd.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)
            unique_counts = rdd.distinct().map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)                      
            stats = total_counts.join(unique_counts)                        
            metrics = stats.mapValues(lambda x: (x[0], x[1], x[1]/x[0]))          
            suspicious_cameras = metrics.filter(lambda x: x[1][0] > MIN_TRAFFIC_COUNT and x[1][2] < MAX_DIVERSITY_RATIO) \
                                        .sortBy(lambda x: x[1][2], ascending=True) \
                                        .take(10)
            
            print(f"========= Time: {time} =========")
            print(f"Cameras with HIGH Traffic (> {MIN_TRAFFIC_COUNT}) and LOW Diversity (< {MAX_DIVERSITY_RATIO}):")
            print(f"{'Camera':<15} | {'Total':<10} | {'Unique':<10} | {'Ratio':<10} | {'Status'}")
            print("-" * 75)
            
            if not suspicious_cameras:
                print("No cameras match the criteria in this window.")
            else:
                for cam, (total, unique, ratio) in suspicious_cameras:
                    print(f"{cam:<15} | {total:<10} | {unique:<10} | {ratio:.4f}     | CONGESTED/LOOP")
            
            print("\n")
    except:
        pass

windowed_dstream.foreachRDD(analyze_camera_behavior)

ssc.start()